# 07 — Problem 1B: Late vs the Promised Date

The brief's Example 4 asks for one underlying problem framed twice: a precise
estimate for planning, and a simple flag for triage. There are two ways to
produce the flag, and this notebook compares them head to head.

1. **Direct.** Train a classifier on the binary label `delivered after the
   promised date`.
2. **Derived.** Take the *regression* from notebook 06, subtract the promised
   date from its prediction, and map that margin to a probability.

On the forward test window the second wins by a factor of two and a half, and the
reason is worth stating: the binary label is 7.4% positive and throws away
everything about *how* late an order was, while the continuous target keeps it.
Thresholding a good regression beats training a classifier on a coarsened version
of the same information.

But the in-training windows do **not** show that gap — they rank the two within
noise of each other, and marginally prefer a direct classifier. Section 7.6 is
about that disagreement, which is the more interesting half of this notebook.

The derived route is the deliverable regardless, and by construction rather than
by measurement: Problem 1 was defined as the brief's dual framing, so the flag
*is* the regression's own answer compared against the promised date. The five
direct classifiers are a control arm.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config, evaluate as E, modeling as M
from data_load import load_all, load_analysis
from features import *
from viz import use_report_style, save_fig

use_report_style()
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "report" / "results"
def results(name):
    """Read one canonical result table written by src/train_models.py."""
    p = RESULTS / name
    assert p.exists(), f"{p} missing - run `python src/train_models.py` first"
    return pd.read_csv(p)
print("seed", config.SEED, "| window", config.ANALYSIS_START.date(), "->", config.ANALYSIS_END.date())

seed 42 | window 2017-01-01 -> 2018-08-31


In [2]:
lat = results("clf_p1b.csv")
display(lat[["model", "family", "route", "val_pr_auc", "roc_auc", "pr_auc",
             "pr_auc_lift", "balanced_accuracy", "precision", "recall", "f1",
             "accuracy", "majority_accuracy", "threshold"]].round(3))

,model,family,route,val_pr_auc,roc_auc,pr_auc,pr_auc_lift,balanced_accuracy,precision,recall,f1,accuracy,majority_accuracy,threshold
0,LogisticRegression,Linear,direct,0.172,0.529,0.092,1.237,0.516,0.112,0.090,0.100,0.880,0.926,0.14
1,LogisticRegression (tuned L2),Linear,direct,0.172,0.512,0.089,1.204,0.517,0.106,0.104,0.105,0.869,0.926,0.17
2,DecisionTreeClassifier,Tree-based,direct,0.145,0.545,0.082,1.103,0.481,0.066,0.262,0.105,0.669,0.926,0.10
3,RandomForestClassifier,Tree-based,direct,0.162,0.527,0.075,1.009,0.499,0.073,0.114,0.089,0.827,0.926,0.14
4,HistGradientBoostingClassifier,Tree-based,direct,0.186,0.584,0.092,1.238,0.524,0.088,0.279,0.134,0.733,0.926,0.11
5,Derived: regression margin vs promise,Derived,derived,0.166,0.782,0.214,2.888,0.705,0.137,0.826,0.235,0.601,0.926,0.11
6,Derived: predicted days > promised days,Derived,derived,NaN,0.569,0.109,1.473,0.569,0.274,0.175,0.214,0.905,0.926,0.50


## 7.1 Accuracy is the wrong headline, demonstrated

Every row carries `majority_accuracy` beside `accuracy` so the two cannot be read
apart. A rule that always predicts "on time" scores about 0.93 and catches nothing.

In [3]:
floor = lat["pr_auc_floor"].iloc[0]
print(f"positive rate in the test window : {floor:.3f}")
print(f"always-predict-on-time accuracy  : {lat['majority_accuracy'].iloc[0]:.3f}")
print(f"no-skill PR-AUC                  : {floor:.3f}\n")
for _, r in lat.iterrows():
    print(f"  {r['model']:44s} acc {r['accuracy']:.3f}  "
          f"balacc {r['balanced_accuracy']:.3f}  PR-AUC {r['pr_auc']:.3f} "
          f"({r['pr_auc_lift']:.1f}x floor)")

positive rate in the test window : 0.074
always-predict-on-time accuracy  : 0.926
no-skill PR-AUC                  : 0.074

  LogisticRegression                           acc 0.880  balacc 0.516  PR-AUC 0.092 (1.2x floor)
  LogisticRegression (tuned L2)                acc 0.869  balacc 0.517  PR-AUC 0.089 (1.2x floor)
  DecisionTreeClassifier                       acc 0.669  balacc 0.481  PR-AUC 0.082 (1.1x floor)
  RandomForestClassifier                       acc 0.827  balacc 0.499  PR-AUC 0.075 (1.0x floor)
  HistGradientBoostingClassifier               acc 0.733  balacc 0.524  PR-AUC 0.092 (1.2x floor)
  Derived: regression margin vs promise        acc 0.601  balacc 0.705  PR-AUC 0.214 (2.9x floor)
  Derived: predicted days > promised days      acc 0.905  balacc 0.569  PR-AUC 0.109 (1.5x floor)


## 7.2 Direct against derived

The comparison has to be like for like, which took one correction. The direct
classifiers arrive with per-fold cross-validated scores from their own
hyperparameter search; the derived route had none, because it is not a fitted
classifier at all. Scoring it only on the validation period and the direct models
on everything would not have been a comparison. So the selected regressor is
refitted inside each cross-validation fold and its margin scored there, giving
both routes the same six windows.

That correction matters, and not in the direction we expected. Scored on the same
six windows, the derived route sits mid-pack — the in-training windows and the
validation period do not separate it from the direct classifiers at all. The gap
only opens on the forward window. Section 7.6 takes that seriously rather than
quietly selecting on the test set.

In [4]:
direct = lat[lat["route"] == "direct"]
derived = lat[lat["route"] == "derived"]
bd = direct.loc[direct["val_pr_auc"].idxmax()]
bv = derived.loc[derived["val_pr_auc"].idxmax()]

comp = pd.DataFrame([
    {"route": "best direct classifier", "model": bd["model"], "ROC-AUC": bd["roc_auc"],
     "PR-AUC": bd["pr_auc"], "lift": bd["pr_auc_lift"],
     "balanced acc": bd["balanced_accuracy"], "F1": bd["f1"]},
    {"route": "derived from the regression", "model": bv["model"], "ROC-AUC": bv["roc_auc"],
     "PR-AUC": bv["pr_auc"], "lift": bv["pr_auc_lift"],
     "balanced acc": bv["balanced_accuracy"], "F1": bv["f1"]},
]).set_index("route")
display(comp.round(3))
print(f"\nderived - direct:  ROC {bv['roc_auc'] - bd['roc_auc']:+.3f},  "
      f"PR-AUC x{bv['pr_auc'] / bd['pr_auc']:.2f}")
assert bv["pr_auc"] > bd["pr_auc"], "the derived route was expected to win"

import figures_phase2 as F
print(Path(F.fig_late_routes()).name)

,model,ROC-AUC,PR-AUC,lift,balanced acc,F1
route,,,,,,
best direct classifier,HistGradientBoostingClassifier,0.584,0.092,1.238,0.524,0.134
derived from the regression,Derived: regression margin vs promise,0.782,0.214,2.888,0.705,0.235



derived - direct:  ROC +0.198,  PR-AUC x2.33


fig12_late_routes.pdf


## 7.3 Why the margin needs calibrating

`predicted_days − promised_days` is a score, not a probability: it ranks orders
correctly but its scale means nothing. The operations team needs a risk it can put
a cost against, so the margin is mapped through a one-parameter logistic fitted on
the **validation** period (Platt, 1999). The test window is untouched by that fit.

In [5]:
import json
meta = json.loads((config.MODELS_DIR / "model_metadata.json").read_text("utf-8"))
p1b = meta["p1b"]
print(f"route selected     : {p1b['route']}  ({p1b['label']})")
print(f"margin -> risk     : p = sigmoid({p1b['platt_coef']:.4f} * margin_days "
      f"{p1b['platt_intercept']:+.4f})")
print(f"decision threshold : {p1b['threshold']:.2f}, tuned on validation only")

pred = results("predictions_p1b_test.csv")
buckets = pd.cut(pred["p_late_derived"], [0, .1, .2, .3, .5, .7, 1.0])
cal = pred.groupby(buckets, observed=True).agg(
    n=("actual_late", "size"), predicted=("p_late_derived", "mean"),
    observed=("actual_late", "mean"))
display(cal.round(3))
print("\nThe two right-hand columns should track each other; where they do, the "
      "risk can be multiplied by a cost.")

route selected     : derived  (Derived: regression margin vs promise)
margin -> risk     : p = sigmoid(0.2285 * margin_days -0.7473)
decision threshold : 0.11, tuned on validation only


,n,predicted,observed
p_late_derived,,,
"(0.0, 0.1]",6481,0.047,0.022
"(0.1, 0.2]",3674,0.144,0.072
"(0.2, 0.3]",1679,0.243,0.180
"(0.3, 0.5]",745,0.361,0.283
"(0.5, 0.7]",45,0.583,0.267
"(0.7, 1.0]",6,0.731,0.500



The two right-hand columns should track each other; where they do, the risk can be multiplied by a cost.


## 7.4 The zero-parameter rule

Worth reporting because it needs no fitting at all: flag an order whenever the
model expects it to arrive after the promised date. It is the rule an operations
team would write on a whiteboard, and it is the honest floor the calibrated
version has to beat.

In [6]:
naive = lat[lat["model"].str.contains("predicted days")]
if len(naive):
    r = naive.iloc[0]
    print(f"  flag when predicted > promised:  balacc {r['balanced_accuracy']:.3f}  "
          f"precision {r['precision']:.3f}  recall {r['recall']:.3f}  F1 {r['f1']:.3f}")
    print(f"  calibrated + tuned threshold  :  balacc {bv['balanced_accuracy']:.3f}  "
          f"precision {bv['precision']:.3f}  recall {bv['recall']:.3f}  F1 {bv['f1']:.3f}")

  flag when predicted > promised:  balacc 0.569  precision 0.274  recall 0.175  F1 0.214
  calibrated + tuned threshold  :  balacc 0.705  precision 0.137  recall 0.826  F1 0.235


## 7.6 When in-sample selection cannot see the answer

Here is the uncomfortable result. Scored on all six out-of-sample windows, the
rolling-origin rule that chose well for Problems 1A and 2 ranks a *direct*
classifier first — and that classifier reaches barely more than the no-skill
floor on the forward window, while the derived route reaches nearly three times
it.

Look at the per-window scores below before the ranks. Every candidate sits within
noise of every other on every in-training window; in the fifth fold they span
0.206 to 0.362 and the ordering is close to arbitrary. The rule has almost
nothing to separate them on, so it separates them on noise.

Why the forward window disagrees is visible in the monthly series. The later
training folds and the validation period are dominated by the February–April 2018
congestion — 16.0% and 21.4% late — and by the May strike. A classifier can fit a
regime. The derived route encodes a *mechanism*: predicted duration against the
promised date. Mechanisms transfer; fitted regimes do not.

We report this rather than presenting the rule as having chosen the derived
route, and we note that confirming the preference properly would need a further
forward window we do not have.

In [7]:
sel = results("selection_p1b.csv")
cols = ["model"] + [c for c in sel.columns if c.startswith("rank_")] +        ["mean_rank_all", "mean_rank"]
display(sel[cols].round(2))
print(f"windows voting: {sel['windows_voting'].iloc[0]}")

sc = [c for c in sel.columns if c.startswith("score_")]
print("per-window PR-AUC — note how little separates them:")
display(sel[["model"] + sc].round(4))

protocol_pick = sel.iloc[0]["model"]
shipped = "Derived: regression margin vs promise"
print(f"the window rule ranks first : {protocol_pick}")
print(f"the deliverable is          : {shipped}")
print("\non the forward test window:")
for m in (protocol_pick, shipped):
    r = lat[lat["model"] == m].iloc[0]
    print(f"  {m:42s} ROC {r['roc_auc']:.3f}  PR-AUC {r['pr_auc']:.3f} "
          f"({r['pr_auc_lift']:.2f}x floor)  balacc {r['balanced_accuracy']:.3f}")

spread = sel[sc].max() - sel[sc].min()
print(f"\nspread between best and worst candidate, per window:")
print(spread.round(3).to_string())

,model,rank_fold1,rank_fold2,rank_fold3,rank_fold4,rank_fold5,rank_validation,mean_rank_all,mean_rank
0,LogisticRegression (tuned L2),2.0,5.0,3.0,2.0,1.0,2.0,2.50,1.67
1,HistGradientBoostingClassifier,4.0,2.0,2.0,1.0,4.0,1.0,2.33,2.00
2,LogisticRegression,5.0,3.0,5.0,5.0,2.0,3.0,3.83,3.33
3,Derived: regression margin vs promise,3.0,4.0,4.0,6.0,3.0,4.0,4.00,4.33
4,RandomForestClassifier,1.0,1.0,1.0,3.0,5.0,5.0,2.67,4.33
5,DecisionTreeClassifier,6.0,6.0,6.0,4.0,6.0,6.0,5.67,5.33


windows voting: fold4, fold5, validation
per-window PR-AUC — note how little separates them:


,model,score_fold1,score_fold2,score_fold3,score_fold4,score_fold5,score_validation
0,LogisticRegression (tuned L2),0.0751,0.1277,0.2477,0.3385,0.3615,0.1724
1,HistGradientBoostingClassifier,0.0622,0.1351,0.2798,0.3507,0.3438,0.1856
2,LogisticRegression,0.0602,0.1321,0.2156,0.2890,0.3550,0.1716
3,Derived: regression margin vs promise,0.0721,0.1281,0.2454,0.2691,0.3453,0.1660
4,RandomForestClassifier,0.0885,0.1359,0.2824,0.3314,0.3205,0.1619
5,DecisionTreeClassifier,0.0574,0.1100,0.2128,0.2956,0.2057,0.1449


the window rule ranks first : LogisticRegression (tuned L2)
the deliverable is          : Derived: regression margin vs promise

on the forward test window:
  LogisticRegression (tuned L2)              ROC 0.512  PR-AUC 0.089 (1.20x floor)  balacc 0.517
  Derived: regression margin vs promise      ROC 0.782  PR-AUC 0.214 (2.89x floor)  balacc 0.705

spread between best and worst candidate, per window:
score_fold1         0.031
score_fold2         0.026
score_fold3         0.070
score_fold4         0.082
score_fold5         0.156
score_validation    0.041


## 7.7 What this means for the model budget

The derived route costs **no extra model**. Problem 1 is served end to end by the
single regression fitted in notebook 06 plus a two-parameter logistic — which is
why the family budget is spent on Problem 2 rather than on a second, weaker
classifier that would have been discarded anyway.